<a href="https://colab.research.google.com/github/rendzina/BigDataAndVisualisation/blob/main/Colab/MongoDB_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MongoDB and Python

*MKU: Big Data and Visualisation*

This notebook shows how to use **MongoDB** from **Python** in Google Colab.

**How it is organised (follow top to bottom):**

1. **Install and import** PyMongo so Python can talk to MongoDB.
2. **Connect to Atlas** (build a URI, create a `MongoClient`). You need a live connection before any data access works.
3. **Choose database and collection** (`db` and `collection` variables for this practical).
4. **Read with `find`**: filter documents and inspect results. A cursor is returned; materialise it with `list(...)` if you want all matching rows at once.
5. **Create, read, update, and delete** on one throwaway demo row (`insert_one`, `find_one`, `update_one`, `delete_one`) so the shared `noise_mapping` data stays intact. The module slide uses the acronym **CRUD** for this set of actions.
6. **Aggregation**: a read-only pipeline example (`$match`, `$sort`, `$project`, `$limit`) for summaries on the server.

In [ ]:
# Colab does not ship with pymongo; this line downloads and installs it into the runtime.
!pip install pymongo

In [ ]:
# Load the pymongo driver so Python can speak the MongoDB wire protocol.
import pymongo
# Import the class used to open a connection and reach databases and collections.
from pymongo import MongoClient

### Before you explore data

You must **connect** first. Run the cells that install PyMongo, import it, build the **connection string**, create **`MongoClient`**, and set **`db`** and **`collection`**. Until those steps succeed, queries and updates have nowhere to go. After connection, work through **Read with `find`**, then the **demo insert / update / delete** cells in order, then **Aggregation** if you are following the notebook from the top.

**MongoDB Atlas (connection cell below):** The code builds a `mongodb+srv://` URI using variables `username`, `cluster`, and `connection_string`. The username is **`bdv_student`** in this practical. Colab will **prompt you for the password**; it is not saved in the notebook. You can change **`cluster`** to your own Atlas hostname if you use a different cluster. Special characters in the password are encoded safely for the URI.

In [ ]:
# Connection to MongoDB Atlas (password is prompted so it is not stored in the notebook)
# getpass hides keyboard input and returns the typed string to the variable (no echo in the log).
import getpass
# quote_plus turns the password into URL-safe text (needed if the password contains @, :, /, etc.).
from urllib.parse import quote_plus

# Atlas database user for this practical (change if you use your own user).
username = "bdv_student"
# Ask for the password interactively; it never appears as a literal in the notebook file.
password = getpass.getpass("Enter MongoDB password for bdv_student: ")

# Hostname of your Atlas cluster (Atlas UI: Connect, then Drivers; copy the host part of the URI).
cluster = "cluster0.nuo8v2x.mongodb.net"

# f-string plugs username and encoded password into the standard mongodb+srv URI pattern.
# The query string asks for sensible default write concern for Atlas.
connection_string = (
    f"mongodb+srv://{username}:{quote_plus(password)}@{cluster}/"
    "?retryWrites=true&w=majority"
)

# Open one client object; it manages the connection pool to the cluster.
client = MongoClient(connection_string)


In [ ]:
# client.<name> returns a Database object for that database name on the connected cluster.
db = client.environmental
# db.<name> returns a Collection object (the set of documents you will query and update).
collection = db.noise_mapping

### Read: query with `find`

After install, connect, and choosing `collection`, you can read data. **`find(filter)`** returns every document that matches the filter dict. PyMongo gives you a **cursor** (a lazy iterator). Run **`list(cursor)`** when you want the full result set in memory as a list of dicts. That is fine for small teaching examples; be careful with huge collections.

In [ ]:
# Dict of field name to value: equality match on that field (same idea as SQL WHERE col = val).
mk_filter = {"Location/Agglomeration": "Milton Keynes Urban Area"}
# find runs the query on the server and returns a Cursor (lazy stream), not a full list yet.
cursor = collection.find(mk_filter)

In [ ]:
# list(...) drains the cursor and builds an in-memory list of all matching documents.
# Fine for teaching-sized results; avoid doing this on millions of rows without limits.
list(cursor)

### Demo row: insert, update, and delete

You already read existing rows with `find` and `list` above. The cells below walk through one **fictional** document in `noise_mapping`: insert it, read it by `_id`, change it with `$set`, read it again, then delete it so the collection stays tidy.

Run the code cells **in order** after the connection cells. **Aggregation** (later) runs a server-side pipeline; it is a different pattern from editing single documents by hand.

In [ ]:
# --- Create: insert_one ---
# insert_one sends one BSON document to the server; MongoDB assigns _id unless you supply it.

# String we reuse as the place name so the row is obviously demo data (not a real agglomeration).
demo_filter_label = "BDV notebook demo (Colab)"
# delete_many removes every document matching the filter (cleans up if you re-run this cell).
collection.delete_many({"Location/Agglomeration": demo_filter_label})

# Python dict becomes one MongoDB document; keys are field names, values are numbers or strings.
demo_doc = {
    "Location/Agglomeration": demo_filter_label,
    "AgglomerationPopulation": 12345,
    "Road_Pop_Lden>=55dB": 100,
}
# insert_result carries metadata from the server, including the new document's primary key.
insert_result = collection.insert_one(demo_doc)
# Save ObjectId for the next cells (update and delete target this exact row).
demo_id = insert_result.inserted_id
print("inserted_id:", demo_id)

In [ ]:
# --- Read: find_one ---
# find_one applies the same filter idea as find, but stops after the first match (or returns None).

# Filter by primary key: the fastest and clearest way to fetch the row you just inserted.
collection.find_one({"_id": demo_id})

In [ ]:
# --- Update: update_one with an update operator ---
# First argument: which document(s) to consider. Second argument: update operators (not a full replacement doc).

collection.update_one(
    # Match exactly our demo row by its ObjectId.
    {"_id": demo_id},
    # $set adds or overwrites the Notes field; other fields on the document are left unchanged.
    {"$set": {"Notes": "Field added during the update step (notebook demo)"}},
)

In [ ]:
# --- Read again: confirm the update ---
# Same query as before; the returned dict should now include the Notes field from $set.

collection.find_one({"_id": demo_id})

In [ ]:
# --- Delete: delete_one ---
# delete_one removes at most one document that matches the filter (here, exactly our demo row).

collection.delete_one({"_id": demo_id})

In [ ]:
# --- Read after delete: expect None ---
# The same _id should no longer exist; find_one therefore returns None (nothing to display in Colab).

collection.find_one({"_id": demo_id})

### Aggregation (pipeline on the server)

**Aggregation** sends a *pipeline* of stages to MongoDB. Each stage transforms the stream of documents, so you can filter, group, sort, and reshape data **before** it comes back to Python.

The example below is **read-only**: it summarises a few existing `noise_mapping` rows by sorting on population and projecting a small set of fields.

In [ ]:
# --- Aggregation: match, then sort, then project, then limit (stages run top to bottom) ---

pipeline = [
    # Keep documents that have a positive population figure (drops missing or zero).
    {"$match": {"AgglomerationPopulation": {"$exists": True, "$gt": 0}}},
    # Sort descending on population so the largest agglomerations appear first (-1 means descending).
    {"$sort": {"AgglomerationPopulation": -1}},
    {
        # Reshape each document: 1 means include field, 0 means omit _id from the output.
        "$project": {
            "_id": 0,
            "Location/Agglomeration": 1,
            "AgglomerationPopulation": 1,
        },
    },
    # Stop after 10 documents so the notebook output stays short.
    {"$limit": 10},
]

# aggregate runs the whole pipeline on the server and returns a cursor; list(...) materialises it.
list(collection.aggregate(pipeline))